# CSC14120 – Parallel Programming – Final Project
# Midterm submission: Proposal and Sequential Version (V0)
## Panorama image stitching — Group 09

## 3.1 Team information and work plan

| Student ID | Full name |
|---|---|
| 23127004 | Lê Nhật Khôi |
| 23127157 | Nguyễn Hoàng Gia Bảo |
| 23127165 | Nguyễn Hải Đăng |

**Work plan until the final submission.** Each member owns two stages through all versions (V0 → V1 → V2/V3): the owner writes the CUDA kernels of these stages, profiles and optimizes them, and presents and answers questions about them in the oral defense. The stages are distributed by the difficulty of their GPU implementation: the two least regular stages (variable-size keypoint compaction + top-K, RANSAC with reductions) go to Khôi, the keypoint-level stages to Bảo, and the per-pixel stages to Đăng, who also wrote the framework and V0 for the midterm.

| Work item | Lê Nhật Khôi | Nguyễn Hoàng Gia Bảo | Nguyễn Hải Đăng |
|---|---|---|---|
| Framework (`main.cu`, `versions.h`, timing, JSON output), dataset tools, V0 (midterm) | review of stages 2, 5 | review of stages 3, 4 | **owner** |
| Stage 1 – preprocessing (gray + separable Gaussian) | | | **owner** (V1, V2+) |
| Stage 2 – keypoint detection (FAST, NMS, compaction, top-K) | **owner** (V1, V2+) | | |
| Stage 3 – description (orientation + rotated BRIEF) | | **owner** (V1, V2+) | |
| Stage 4 – matching (Hamming + ratio test) | | **owner** (V1, V2+) | |
| Stage 5 – homography (RANSAC + refinement) | **owner** (V1, V2+) | | |
| Stage 6 – warping and blending | | | **owner** (V1, V2+) |
| Optimizations (one main idea per version) | prefix-sum compaction, one warp per RANSAC hypothesis | shared-memory tiling of descriptors + `__popc`, CUDA streams for images A and B | shared-memory tiles with halos + constant memory for the blur, gray + blur fusion, texture memory for the warp |
| Benchmark runs, speedup tables, final comparison | | | **owner** |
| Profiling (Nsight Systems / Compute) | stages 2, 5 | stages 3, 4 | stages 1, 6 |
| Dataset capture (3 sites + challenging pair) | ✔ | ✔ | ✔ |
| Report notebook, slides, video | own stages | own stages | own stages + overall structure |

**Dataset:** raw photos + prepared `dataset/` folder: ⏳ *Google Drive link to be added*.

## Setup

The notebook builds the program from the submitted `src/` folder (single source of truth). Put the team's Drive folder `Group09_Midterm/` (containing `src/` and `raw_photos/`) in **My Drive**, or upload these two folders to `/content/work`. Then *Runtime → Run all* on a GPU runtime.

In [ ]:
import os, shutil, glob, json
PROJECT = '/content/drive/MyDrive/Group09_Midterm'   # team's shared Drive folder
WORK = '/content/work'
os.makedirs(WORK, exist_ok=True)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    for d in ['src', 'raw_photos']:
        if os.path.isdir(os.path.join(PROJECT, d)):
            shutil.copytree(os.path.join(PROJECT, d), os.path.join(WORK, d), dirs_exist_ok=True)
except Exception as e:
    print('Drive not used:', e)
os.chdir(WORK)
assert os.path.isdir('src') and os.path.isdir('raw_photos'), 'src/ and raw_photos/ must be in ' + WORK
print(sorted(os.listdir('src')))
print(sorted(os.listdir('raw_photos')))

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!nvcc --version | tail -n 2
!lscpu | grep -E "Model name|^CPU\(s\)|Thread\(s\) per core"

In [ ]:
# Build: one program for all versions (only V0 exists at the midterm)
!nvcc -O2 -std=c++17 -arch=native src/main.cu src/v0_orb.cpp src/stb_impl.cpp -o stitch
!./stitch 2>&1 | head -n 1

## 3.2 Problem understanding

A panorama is built from two photos taken from the **same position** while the camera is **rotated** between the shots. Because the optical center does not move, every scene point seen in both photos is related by a single 3×3 projective transform, the homography **H** (for a pure rotation R with intrinsics K: H = K R K⁻¹). If we know H, we can re-project photo B into the frame of photo A and merge them into one wide image. If the photographer walks sideways instead, near and far objects shift by different amounts (parallax) and no single homography can align the whole image, which is why the photos must be taken by rotating, not walking.

The problem is that H is unknown, so the pipeline estimates it from the images themselves:

1. **Preprocessing** – convert both RGB images to grayscale and smooth them, because the following steps only use intensity and must not react to pixel noise.
2. **Keypoint detection** – find points that can be located precisely in both images, i.e. corners (an edge can slide along itself, a corner cannot). We keep the K strongest ones.
3. **Description** – summarize the neighbourhood of each keypoint in a compact vector (here 256 bits) that is the same for the same physical point in both photos, even if the patch is slightly rotated.
4. **Matching** – for each keypoint of A, find the most similar descriptor in B; matches whose best candidate is not clearly better than the second best are ambiguous (repetitive texture) and are rejected.
5. **Homography estimation** – even after filtering, some matches are wrong (outliers). RANSAC repeatedly fits H to 4 random matches and keeps the hypothesis that agrees with the most matches (inliers); H is then refined by least squares on all inliers.
6. **Warping and blending** – for each pixel of the output canvas, look up where it comes from in A and (through H⁻¹) in B, sample B with bilinear interpolation, and blend the overlap so that no seam is visible.

Stages 1–3 are applied to each image independently (they could run concurrently, e.g. in two CUDA streams); stages 4–6 need both images. Stages 1, 2 and 6 work on every pixel (millions of independent tasks), stages 3–5 work on keypoints, matches and hypotheses (thousands of independent tasks), which is what makes the problem suitable for the GPU.

## 3.3 Chosen methods

| Stage | Method (V0 and all later versions) | Main parameters (`src/common.h`) |
|---|---|---|
| 1. Preprocessing | Gray = 0.299R + 0.587G + 0.114B; **separable Gaussian blur** (horizontal then vertical 1D pass), border = nearest pixel | σ = 1.5, 9 taps (radius 4) |
| 2. Detection | **FAST-9** segment test on the gray image (9 contiguous pixels of the 16-pixel circle brighter/darker than the center ± t), score = sum of absolute differences, **3×3 non-maximum suppression**, **top-K** by score | t = 20, K = 2000 per image, border = 20 px |
| 3. Description | **ORB-style**: orientation from the intensity centroid of a circular patch, **rotated BRIEF** with 256 fixed random point pairs on the blurred image | patch radius 15, 256 bits, pairs in [−13, 13]², σ = 31/5 |
| 4. Matching | **Brute force** A → B with **Hamming distance** (XOR + popcount), **Lowe's ratio test** | ratio = 0.8, max distance = 80 bits |
| 5. Homography | **RANSAC** with the **4-point DLT** (8×8 linear system), then **least-squares refinement** on the inliers (Hartley normalization), 3 iterations | 2000 hypotheses (fixed), threshold 3 px at 1280 px width (scaled with the width), min 12 inliers |
| 6. Warp + blend | **Inverse warping** (one gather per output pixel), **bilinear** sampling of B, **feathering** (weight = distance to the image border) | canvas limited to 4× the size of A |

**Justification (quality and GPU suitability).**

- *Separable Gaussian*: same result as the 2D filter but 2·9 = 18 instead of 81 multiplications per pixel; it is a classic stencil (one thread per pixel, shared-memory tiles with halos later).
- *FAST + top-K*: the segment test only compares integers, so it is cheap and every pixel is independent. Fixing K makes the work of stages 3–5 predictable and identical across versions. On the GPU, collecting a variable number of corners needs compaction (atomics or a prefix sum, lectures 08–09), and top-K needs a sort; we break ties deterministically (score, then y, then x) so that every version keeps exactly the same keypoints.
- *ORB / rotated BRIEF*: a 256-bit binary descriptor is computed with 256 intensity comparisons and compared with 8 XOR + popcount instructions (`__popc` on the GPU), much cheaper than SIFT's float histograms. Our pairs of photos have the same scale (rotation only, same camera), so we do not need SIFT's scale invariance; the orientation handles the small roll between hand-held shots.
- *Brute-force matching*: 2000 × 2000 = 4 million distances, a regular all-pairs computation similar to matrix multiplication (tiling in shared memory). The ratio test removes most matches on repetitive textures.
- *RANSAC with a fixed number of pre-generated samples*: hypotheses are independent (one thread / warp per hypothesis on the GPU), and generating the random samples once on the host makes every version test exactly the same hypotheses, so their results can be compared with V0. The refinement on all inliers is essential: a homography from only 4 points is noisy and its error grows towards the far corners of B.
- *Inverse warping + feathering*: every output pixel is computed independently (a gather, no write conflicts); bilinear sampling can later use texture memory. Feathering removes visible seams caused by exposure differences, at the cost of slight ghosting if the alignment is imperfect.

## 3.4 Sequential version (V0)

**Code structure** (`src/`):

| File | Content |
|---|---|
| `common.h` | data types (images, keypoints, 256-bit descriptors, matches), **all parameters** (`Params`), per-stage timing structure |
| `versions.h` | common signature `stitchVx(imgA, imgB, params, result)` and version selection (`--version N`) |
| `main.cu` | command line `./stitch <left> <right> <out> --version N --runs N [--gt H.txt] [--json f] [--dump f]`: loads images (not timed), 1 warm-up run + N timed runs, per-stage average/min, homography error against a ground truth, JSON output for this notebook |
| `v0_orb.cpp` | the sequential pipeline: one function per stage, each stage timed with `std::chrono::high_resolution_clock` |
| `cuda_utils.cuh` | `CHECK` macro, `GpuTimer` (CUDA events), GPU name (used by V1+) |
| `stb_impl.cpp` + `stb_image*.h` | image I/O (stb by Sean Barrett, public domain) |
| `tools/prepare_dataset.py`, `tools/make_synthetic.py` | dataset preparation and synthetic pair with ground truth |

**Which part is done in which language** (as required, only the computational pipeline is parallelized; auxiliary steps use Python in this notebook):

| Part | Language | Where | Timed? |
|---|---|---|---|
| Photo preparation: 16:9 crop, resize to 3 sizes; synthetic pair + ground truth | Python (Pillow, NumPy) | `src/tools/*.py`, run once in this notebook | no |
| Image loading / saving | C (stb_image) | `main.cu`, `stb_impl.cpp` | no (excluded) |
| **Stages 1–6: the whole stitching pipeline** | **C++, sequential (V0)** → **CUDA (V1, V2, …)** | `v0_orb.cpp` → `v1_*.cu`, `v2_*.cu`, … | **yes, per stage** |
| Host-side setup shared by all versions (Gaussian weights, BRIEF pattern, RANSAC samples) | C++ on the host | `v0_orb.cpp` | yes (measured separately, < 0.1 ms) |
| Tables, charts, figures, cross-check with OpenCV (validation only) | Python in this notebook | this notebook | no |

**Fair baseline:** V0 is single-threaded C++ compiled with `-O2`: no OpenMP, no `std::thread`, no hand-written SIMD. Image loading and saving are excluded from the timings, as required.

**Design choices that prepare the CUDA versions:**
- Images are flat row-major arrays (`uint8_t`), keypoints/descriptors/matches are plain structs in contiguous arrays, so they can be copied to the device as they are.
- The BRIEF pattern and the RANSAC samples are generated deterministically from fixed seeds on the host, so V1+ can copy them to the device (constant / global memory) and produce the same keypoints, matches and homography as V0.
- The RANSAC threshold is scaled with the image width, so the three resolutions of the same scene are processed with the same parameters in pixels of the original scene.

**Reused / adapted code:** the `CHECK` macro and `GpuTimer` come from the course demo code (`01-AddVector.cu`); stb_image / stb_image_write (public domain) for I/O. All algorithms (FAST, rotated BRIEF, ratio test, RANSAC/DLT with normalization, bilinear warping, feathering) were written by the team from the papers listed in the References; no code was copied from OpenCV.

Short excerpt – the FAST arc test uses a 16-bit mask per candidate, which is branch-light and will map well to one GPU thread per pixel:
```cpp
uint32_t m = mask16 | (mask16 << 16);   // unroll the circle
uint32_t run = m;
for (int i = 1; i < 9; i++) run &= m >> i;   // != 0  <=>  9 consecutive pixels
```

## 3.6 Dataset

Raw photos are in `raw_photos/<pair>/` (2 photos per pair, the left photo first in alphabetical order, or named `left.*` / `right.*`); pairs whose name starts with `challenging` are the challenging pairs. `tools/prepare_dataset.py` crops every photo to 16:9 (largest centred crop, EXIF orientation applied) and resizes it to the three sizes: **small 1280×720, medium 1920×1080, large 3840×2160**.

The **synthetic pair** is generated by `tools/make_synthetic.py` from one of our own photos: image A is a window of the photo, image B is rendered as the view of the same camera turned by 20° (yaw) with 1.5° pitch and 2° roll. For a rotating camera the two images are related exactly by H = K R K⁻¹, which is saved as the ground truth `H_gt.txt` for each resolution. The ground truth therefore comes from a **different method** than the one being evaluated: it is computed analytically from the chosen camera rotation and used to *render* image B, while our pipeline has to *estimate* H from the image content only (features, matching, RANSAC).

In [ ]:
!pip -q install pillow-heif   # only needed for iPhone HEIC photos

In [ ]:
!python3 src/tools/prepare_dataset.py raw_photos dataset

In [ ]:
# Source photo of the synthetic pair: one of the team's own photos (landscape)
from PIL import Image
try:
    import pillow_heif; pillow_heif.register_heif_opener()
except ImportError:
    pass
# (the largest photo of the standard pairs, so that the large 3840x2160 synthetic pair is not upsampled too much)
cands = [f for f in sorted(glob.glob('raw_photos/*/*')) if not os.path.basename(os.path.dirname(f)).startswith('challenging')]
SYNTH_SOURCE = max(cands, key=lambda f: Image.open(f).size[0] * Image.open(f).size[1])
print('Synthetic pair generated from', SYNTH_SOURCE)
!python3 src/tools/make_synthetic.py "{SYNTH_SOURCE}" dataset/synthetic 20

In [ ]:
# Overview of the prepared pairs (medium size)
import matplotlib.pyplot as plt
from PIL import Image
PAIRS = sorted(p for p in os.listdir('dataset') if os.path.isdir(os.path.join('dataset', p)))
STANDARD = [p for p in PAIRS if p != 'synthetic' and not p.startswith('challenging')]
CHALLENGING = [p for p in PAIRS if p.startswith('challenging')]
print('standard pairs:', STANDARD, '| challenging:', CHALLENGING)
fig, axes = plt.subplots(len(PAIRS), 2, figsize=(12, 3.4 * len(PAIRS)), squeeze=False)
for i, p in enumerate(PAIRS):
    for j, r in enumerate(['A', 'B']):
        axes[i, j].imshow(Image.open(f'dataset/{p}/medium/{r}.jpg'))
        axes[i, j].set_title(f'{p} – {r}')
        axes[i, j].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Keep a copy of the dataset (raw + prepared) in the shared Drive folder for the link above
if os.path.isdir(PROJECT):
    !cd /content/work && zip -qr "{PROJECT}/Group09_dataset.zip" raw_photos dataset && ls -lh "{PROJECT}/Group09_dataset.zip"

## 3.5 V0 results and profiling

All runs below: `--version 0 --runs 5` (1 warm-up run + 5 timed runs, average reported; the minimum is also saved in the JSON files). The CPU and GPU names are printed by the program with every result.

### Correctness on the synthetic pair (homography error against the ground truth)

Error = distance between the 4 corners of B mapped by the estimated H and by H_gt (in pixels of image A).

In [ ]:
import pandas as pd
os.makedirs('results', exist_ok=True)
SIZES = ['small', 'medium', 'large']
import subprocess
def run(pair, size, runs=5, extra=''):
    d = f'dataset/{pair}/{size}'
    out = f'results/{pair}_{size}'
    cmd = f'./stitch {d}/A.jpg {d}/B.jpg {out}.jpg --version 0 --runs {runs} --json {out}.json {extra}'
    log = subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout
    return json.load(open(out + '.json')), log

rows = []
for s in SIZES:
    r, log = run('synthetic', s, extra=f'--gt dataset/synthetic/{s}/H_gt.txt')
    if s == 'small':
        print(log)
    rows.append({'size': s, 'resolution': f"{r['width']}x{r['height']}", 'matches': r['matches'],
                 'inliers': r['inliers'], 'mean corner error (px)': r['gt_corner_error_mean_px'],
                 'max corner error (px)': r['gt_corner_error_max_px']})
print(pd.DataFrame(rows).to_string(index=False, float_format='%.3f'))

⏳ *To be written after the Colab run (every number must come from the outputs above).*

### Timing of every stage at the three resolutions (standard pairs) and panoramas

In [ ]:
records = []
for p in STANDARD + CHALLENGING:
    for s in SIZES:
        r, log = run(p, s)
        rec = {'pair': p, 'size': s, 'ok': r['ok'], 'keypointsA': r['keypointsA'], 'matches': r['matches'],
               'inliers': r['inliers'], 'total_ms': r['total_avg_ms'], 'total_min_ms': r['total_min_ms']}
        rec.update(r['stages_avg_ms'])
        rec['host_setup_ms'] = r['host_setup_avg_ms']
        records.append(rec)
        print(f"{p:24s} {s:6s} ok={r['ok']!s:5s} matches={r['matches']:4d} inliers={r['inliers']:4d} "
              f"total={r['total_avg_ms']:9.1f} ms  {r['message']}")
print('\nGPU:', r['gpu'], '| CPU:', r['cpu'])
df = pd.DataFrame(records)

In [ ]:
STAGES = ['1_preprocess', '2_detect', '3_describe', '4_match', '5_homography', '6_warp_blend']
std = df[df.pair.isin(STANDARD)]
table = std.groupby('size')[STAGES + ['total_ms', 'host_setup_ms']].mean().reindex(SIZES)
print('Average time (ms) of each stage over the standard pairs, V0, average of 5 runs')
print(table.T.to_string(float_format='%.1f'))
pct = table[STAGES].div(table['total_ms'], axis=0) * 100
print('\nShare of the total time (%)')
print(pct.T.to_string(float_format='%.1f'))
print('\nTotal time per pair (ms)')
print(df.pivot(index='pair', columns='size', values='total_ms')[SIZES].to_string(float_format='%.1f'))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
table[STAGES].plot.bar(stacked=True, ax=ax[0], rot=0)
ax[0].set_ylabel('time (ms)'); ax[0].set_title('V0: time per stage (standard pairs, average)')
pct.plot.bar(stacked=True, ax=ax[1], rot=0, legend=False)
ax[1].set_ylabel('% of total'); ax[1].set_title('V0: share of each stage')
plt.tight_layout(); plt.show()

### Panoramas (one per site, medium size)

In [ ]:
for p in STANDARD + CHALLENGING:
    f = f'results/{p}_medium.jpg'
    row = df[(df.pair == p) & (df['size'] == 'medium')].iloc[0]
    plt.figure(figsize=(14, 5))
    if os.path.exists(f) and row.ok:
        plt.imshow(Image.open(f))
    plt.title(f"{p} (medium): {row.inliers} inliers / {row.matches} matches" + ('' if row.ok else ' – FAILED'))
    plt.axis('off'); plt.show()

In [ ]:
# Inlier matches of one standard pair (small size), drawn from the --dump output
import numpy as np
p = STANDARD[0]
d = f'dataset/{p}/small'
!./stitch {d}/A.jpg {d}/B.jpg results/dump.jpg --runs 1 --dump results/dump.txt > /dev/null
lines = open('results/dump.txt').read().split('\n')
def block(tag):
    i = next(k for k, l in enumerate(lines) if l.startswith('# ' + tag))
    n = int(lines[i].split()[2])
    return [list(map(float, l.split())) for l in lines[i + 1:i + 1 + n]]
kA, kB, M = block('keypointsA'), block('keypointsB'), block('matches')
A, B = np.asarray(Image.open(f'{d}/A.jpg')), np.asarray(Image.open(f'{d}/B.jpg'))
plt.figure(figsize=(16, 5)); plt.imshow(np.hstack([A, B]))
for a, b, dist, inl in M:
    if inl:
        xa, ya = kA[int(a)][:2]; xb, yb = kB[int(b)][:2]
        plt.plot([xa, xb + A.shape[1]], [ya, yb], '-', lw=0.4, color='lime')
plt.title(f'{p}: {int(sum(m[3] for m in M))} RANSAC inliers out of {len(M)} matches'); plt.axis('off'); plt.show()

### Cross-check with OpenCV (real pairs, no ground truth)

The real pairs have no ground truth, so we compare our homography with the one estimated by an independent implementation, **OpenCV** (ORB with 5000 features, ratio test 0.75, `findHomography` with RANSAC, threshold 3 px), following the OpenCV feature-homography tutorial. OpenCV is used **only for this validation**, never inside our pipeline. The difference is measured like the ground-truth error: distance between the 4 corners of B mapped by the two homographies (medium size). On the synthetic pair, both are also compared with the true H.

In [ ]:
import cv2, numpy as np
def opencv_homography(a_path, b_path):
    A = cv2.imread(a_path, cv2.IMREAD_GRAYSCALE); B = cv2.imread(b_path, cv2.IMREAD_GRAYSCALE)
    orb = cv2.ORB_create(nfeatures=5000)
    kA, dA = orb.detectAndCompute(A, None); kB, dB = orb.detectAndCompute(B, None)
    knn = cv2.BFMatcher(cv2.NORM_HAMMING).knnMatch(dB, dA, k=2)
    good = [m[0] for m in knn if len(m) == 2 and m[0].distance < 0.75 * m[1].distance]
    if len(good) < 4:
        return None, 0
    src = np.float32([kB[m.queryIdx].pt for m in good]); dst = np.float32([kA[m.trainIdx].pt for m in good])
    H, mask = cv2.findHomography(src, dst, cv2.RANSAC, 3.0)
    return H, int(mask.sum()) if mask is not None else 0

def corner_distance(H1, H2, w, h):
    c = np.float32([[0, 0], [w, 0], [w, h], [0, h]]).reshape(-1, 1, 2)
    d = np.linalg.norm(cv2.perspectiveTransform(c, H1) - cv2.perspectiveTransform(c, H2), axis=2)
    return d.mean(), d.max()

rows = []
for p in ['synthetic'] + STANDARD + CHALLENGING:
    d = f'dataset/{p}/medium'
    r = json.load(open(f'results/{p}_medium.json'))
    H_ours = np.array(r['H']).reshape(3, 3)
    H_cv, cv_inliers = opencv_homography(f'{d}/A.jpg', f'{d}/B.jpg')
    row = {'pair': p, 'ours ok': r['ok'], 'our inliers': r['inliers'], 'OpenCV inliers': cv_inliers}
    if H_cv is not None:
        row['ours vs OpenCV mean (px)'], row['ours vs OpenCV max (px)'] = corner_distance(H_ours, H_cv, r['width'], r['height'])
        if p == 'synthetic':
            G = np.loadtxt(f'{d}/H_gt.txt')
            row['ours vs GT mean (px)'] = corner_distance(H_ours, G, r['width'], r['height'])[0]
            row['OpenCV vs GT mean (px)'] = corner_distance(H_cv, G, r['width'], r['height'])[0]
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False, float_format='%.2f', na_rep='-'))

⏳ *To be written after the Colab run (every number must come from the outputs above).*

### Hot stages

In [ ]:
for s in SIZES:
    order = pct.loc[s].sort_values(ascending=False)
    print(f"{s:6s}: " + ', '.join(f'{k} {v:.1f}%' for k, v in order.items()))
growth = table.loc['large'] / table.loc['small']
print('\nGrowth factor large / small (pixels x 9):')
print(growth.to_string(float_format='%.2f'))

⏳ *To be written after the Colab run (every number must come from the outputs above).*

### Amdahl estimate

If the stages moved to the GPU take a fraction p of V0's time, the end-to-end speedup is at most 1 / (1 − p). V1 must run **every stage** on the GPU; what stays on the CPU is only the host-side setup that every version shares (Gaussian weights, BRIEF pattern, RANSAC samples), measured separately by the program. The table also shows the limit if only the hottest stages were accelerated, which tells us where the effort pays off.

In [ ]:
rows = []
for s in SIZES:
    order = pct.loc[s].sort_values(ascending=False)
    for k in [1, 2, 3, 6]:
        chosen = list(order.index[:k])
        # host-side setup (Gaussian kernel, BRIEF pattern, RANSAC samples) stays on the CPU in every version
        p = (table.loc[s, chosen].sum() - (table.loc[s, 'host_setup_ms'] if k == 6 else 0)) / table.loc[s, 'total_ms']
        rows.append({'size': s, 'accelerated stages': ', '.join(chosen) if k < 6 else 'all 6 stages',
                     'p': p, 'max speedup 1/(1-p)': (1 / (1 - p)) if p < 1 else float('inf')})
print(pd.DataFrame(rows).to_string(index=False, float_format='%.4f'))

⏳ *To be written after the Colab run (every number must come from the outputs above).*

## 3.7 GPU plan

**Which stages go to the GPU.** V1 moves all six stages to the GPU and keeps every intermediate result (gray/blurred images, score map, keypoints, descriptors, matches) in device memory: only the two input images are copied to the device and only the panorama is copied back (HW1 showed that unnecessary transfers cost as much as the kernels themselves). The profiling above decides where the optimization effort goes in V2/V3: ⏳ *hot stages to be confirmed from the measurements*.

**Planned thread mapping for V1 (straightforward kernels, global memory):**

| Stage | V1 kernel(s) | One thread computes | Notes |
|---|---|---|---|
| 1 | `rgbToGray`, `blurRows`, `blurCols` (2D grid of 2D blocks, e.g. 32×8) | one pixel | separable passes, border clamp, Gaussian weights copied to the device |
| 2 | `fastScore` → `nms` → compaction → sort | one pixel (score, NMS); local maxima appended to a candidate list with `atomicAdd` | top-K: sort the candidates by the key (score, y, x) on the device (radix sort, lecture 09), keep the first K – same keypoints as V0 whatever the order of the atomics |
| 3 | `describe` | one keypoint (orientation: ~700 pixel reads, then 256 comparisons) | BRIEF pattern (1 KB) copied once |
| 4 | `match` | one descriptor of A, looping over all 2000 descriptors of B (best + second best, ratio test) | output: one slot per keypoint of A + a valid flag, compacted afterwards (keeps V0's order) |
| 5 | `ransacScore` → argmax reduction → `refine` | one hypothesis: solve its 8×8 system and count the inliers over all matches | samples pre-generated on the host (same hypotheses as V0); refinement: one thread per match builds its normal-equation terms, reduction, 8×8 solve by one thread |
| 6 | `warpBlend` | one output pixel (gather from A and bilinear from B, feather weights) | canvas size and H⁻¹ computed on the host from H (9 numbers) |

**Optimization ideas for V2, V3 (each will be motivated by a measured bottleneck):**
1. *Shared-memory tiling for the stencils* (blur, FAST, orientation patches): load a tile with its halo once per block instead of reading each pixel 9–49 times from global memory; Gaussian weights and BRIEF pattern in **constant memory**; fusing gray conversion with the horizontal blur.
2. *Faster matching*: tile the descriptors of B in shared memory (all threads of a block reuse them), `__popc` on 32-bit words, and possibly one warp per descriptor of A with warp-shuffle reductions for best / second best.
3. *Pipeline level*: process images A and B in two **CUDA streams** for stages 1–3, pinned host memory for the transfers; texture memory for the bilinear sampling of the warp.

**Main risks.**
- *Consistency with V0*: float results computed in a different order (FMA on the GPU) can change a few BRIEF bits or a few RANSAC inlier decisions; we will compare keypoints, matches and the final H with V0 and report the differences, as in HW1.
- *Compaction and top-K* (variable number of corners) are the least regular part: atomics contention, and a sort that is too slow could cancel the speedup of detection.
- *RANSAC on the GPU* has little work per thread (2000 hypotheses × a few hundred matches) and an 8×8 solve per thread (registers, divergence on degenerate samples).
- *Large panoramas*: a 4K pair produces a ~6000×3000 canvas; memory usage and transfer time of the result must be measured.

## References

1. E. Rosten, T. Drummond. *Machine learning for high-speed corner detection*. ECCV 2006 (FAST).
2. M. Calonder, V. Lepetit, C. Strecha, P. Fua. *BRIEF: Binary Robust Independent Elementary Features*. ECCV 2010.
3. E. Rublee, V. Rabaud, K. Konolige, G. Bradski. *ORB: an efficient alternative to SIFT or SURF*. ICCV 2011.
4. D. Lowe. *Distinctive image features from scale-invariant keypoints*. IJCV 2004 (ratio test).
5. M. Fischler, R. Bolles. *Random Sample Consensus*. Communications of the ACM, 1981.
6. R. Hartley, A. Zisserman. *Multiple View Geometry in Computer Vision*, 2nd ed., Cambridge University Press 2004 (DLT, normalization).
7. R. Szeliski. *Computer Vision: Algorithms and Applications*, 2nd ed., Springer 2022 (feature matching, alignment, stitching).
8. S. Nayar. *Image Stitching*, First Principles of Computer Vision monograph FPCV-2-4.
9. CSC14120 lecture slides and demo code (CHECK macro, GpuTimer).
10. stb_image / stb_image_write, S. Barrett, https://github.com/nothings/stb (public domain).

## AI tool usage declaration

We used **Claude (Anthropic)** as an assistant to help draft the code structure and the sequential implementation, the dataset scripts and this notebook, and to explain algorithms (FAST, BRIEF, RANSAC normalization). Every member has reviewed the code of the stages they own and is able to explain and modify it; all timings, images and numbers in this report come from running the submitted code in this notebook.